# 60 — Joint Precision-Boost: Weight × Threshold Grid + FP Feature Suppression

**Builds on**: nb59 (precision-boosting experiments)

**nb59 summary**:
- nb51 REF-CLEAN baseline: F1=0.5128  P=0.459  R=0.581
- Config A (threshold sweep): balanced P=R at t=0.45 → F1=0.4852  P=0.484  R=0.486
- Config B (class weights): 0.5× balanced → best F1=0.5205  P=0.446  R=0.625
- Config B (class weights): 0.25× balanced → best precision P=0.452  R=0.585
- Config E (FP analysis): identified FP-driving features

**This notebook implements the three recommended next steps**:

| Config | Strategy |
|--------|----------|
| A | Joint weight × threshold grid search — tune both levers simultaneously |
| B | FP-feature suppression — drop top FP-driving venue features, retrain |
| C | FP-feature re-weighting — reduce LightGBM feature importance via sample weights |
| D | Combined: best weight from grid + FP feature suppression |

**Goal**: improve precision above 0.484 (Config A balanced point) while keeping F1 ≥ nb51 baseline (0.5128).

In [ ]:
import sys
sys.path.append('../../')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score, recall_score,
    precision_recall_curve, average_precision_score,
)
from lightgbm import LGBMClassifier

RANDOM_STATE = 42
TRAIN_YEARS  = [2015, 2016, 2017]
TEST_YEARS   = [2018, 2019, 2020]

# nb51 REF-CLEAN reference
NB51_F1        = 0.5128
NB51_PRECISION = 0.459
NB51_RECALL    = 0.581
NB51_AUC       = 0.7965

# nb59 Config A balanced point
NB59A_T        = 0.45
NB59A_F1       = 0.4852
NB59A_P        = 0.484
NB59A_R        = 0.486

# nb59 Config B best F1
NB59B_F1       = 0.5205
NB59B_P        = 0.446
NB59B_R        = 0.625

print('Libraries loaded')

## 1. Load data & build features (identical to nb51 / nb59)

In [ ]:
df = pd.read_pickle('../../data/processed/all_unis_cleaned.pkl')
df_aub = df[df['institution'] == 'AUB'].copy()

df_aub_train = df_aub[df_aub['Year'].isin(TRAIN_YEARS)].copy()
df_aub_test  = df_aub[df_aub['Year'].isin(TEST_YEARS)].copy()

print(f'AUB train: {len(df_aub_train):,}  |  AUB test: {len(df_aub_test):,}')

In [ ]:
COL_MAP = {
    'snip':             'SNIP (publication year)',
    'snip_pct':         'SNIP percentile',
    'citescore':        'CiteScore (publication year)',
    'citescore_pct':    'CiteScore percentile',
    'sjr':              'SJR (publication year)',
    'sjr_pct':          'SJR percentile',
    'topic_prom':       'Topic Prominence Percentile',
    'num_authors':      'Authors',
    'num_institutions': 'Affiliations',
    'num_countries':    'Countries',
}
VENUE_FEATS = list(COL_MAP.keys())

def extract_venue_features(subset_df):
    vf = pd.DataFrame(index=subset_df.index)
    for feat, col in COL_MAP.items():
        if col in subset_df.columns:
            if col in ('Authors', 'Affiliations', 'Countries'):
                vf[feat] = subset_df[col].fillna('').apply(
                    lambda x: len(str(x).split(';')) if x else 1
                )
            else:
                vf[feat] = pd.to_numeric(subset_df[col], errors='coerce')
    return vf

def build_features(df_tr, df_te, drop_cols=None):
    """Build TF-IDF + venue feature matrices.
    drop_cols: list of venue feature names to exclude (for FP suppression).
    """
    tfidf = TfidfVectorizer(
        max_features=5000, ngram_range=(1, 2),
        min_df=5, max_df=0.8, stop_words='english'
    )
    prep = lambda s: str(s).lower() if pd.notna(s) else ''
    tr_mat = tfidf.fit_transform(df_tr['Abstract'].apply(prep))
    te_mat = tfidf.transform(df_te['Abstract'].apply(prep))
    cols   = [f'tfidf_{f}' for f in tfidf.get_feature_names_out()]

    tfidf_tr = pd.DataFrame(tr_mat.toarray(), index=df_tr.index, columns=cols)
    tfidf_te = pd.DataFrame(te_mat.toarray(), index=df_te.index, columns=cols)

    vf_tr = extract_venue_features(df_tr)
    vf_te = extract_venue_features(df_te)
    tr_med = vf_tr.median()
    vf_tr  = vf_tr.fillna(tr_med)
    vf_te  = vf_te.fillna(tr_med)

    if drop_cols:
        vf_tr = vf_tr.drop(columns=[c for c in drop_cols if c in vf_tr.columns])
        vf_te = vf_te.drop(columns=[c for c in drop_cols if c in vf_te.columns])

    X_tr = pd.concat([tfidf_tr, vf_tr.set_index(tfidf_tr.index)], axis=1)
    X_te = pd.concat([tfidf_te, vf_te.set_index(tfidf_te.index)], axis=1)
    return X_tr, X_te

# Labels at 75th pct (train-only threshold)
thr_75    = df_aub_train['Citations'].quantile(0.75)
y_tr      = (df_aub_train['Citations'] >= thr_75).astype(int)
y_te      = (df_aub_test['Citations']  >= thr_75).astype(int)

print(f'Train pos rate: {y_tr.mean():.1%}  |  Test pos rate: {y_te.mean():.1%}')
print(f'Citation threshold (75th pct, train): {thr_75:.0f}')

pos_rate       = y_tr.mean()
balanced_weight = (1 - pos_rate) / pos_rate
print(f'balanced class weight = {balanced_weight:.2f}')

print('Building features...')
X_tr, X_te = build_features(df_aub_train, df_aub_test)
print(f'X_tr: {X_tr.shape}  |  X_te: {X_te.shape}')

## 2. Config A — Joint Weight × Threshold Grid

nb59 tuned class weight and decision threshold independently. Here we sweep both
simultaneously to find the Pareto frontier: for each weight setting, find the threshold
that maximises F1, then record the resulting (P, R, F1) operating point.

This answers: *which (weight, threshold) pair gives the best precision without losing F1?*

In [ ]:
weight_scales = [1.0, 0.75, 0.60, 0.50, 0.40, 0.35, 0.25, 0.15]
threshold_grid = np.arange(0.05, 0.95, 0.01)

grid_results = []

print('Running weight × threshold joint grid...')
for scale in weight_scales:
    w = balanced_weight * scale
    cw = {0: 1.0, 1: w}
    model = LGBMClassifier(
        n_estimators=500, learning_rate=0.05, num_leaves=63,
        class_weight=cw, random_state=RANDOM_STATE,
        n_jobs=-1, verbose=-1
    )
    model.fit(X_tr, y_tr)
    proba = model.predict_proba(X_te)[:, 1]
    auc   = roc_auc_score(y_te, proba)

    for t in threshold_grid:
        y_pred = (proba >= t).astype(int)
        if y_pred.sum() == 0:
            continue
        grid_results.append({
            'weight_scale':  scale,
            'pos_weight':    round(w, 3),
            'threshold':     round(t, 2),
            'f1':            f1_score(y_te, y_pred, zero_division=0),
            'precision':     precision_score(y_te, y_pred, zero_division=0),
            'recall':        recall_score(y_te, y_pred, zero_division=0),
            'auc':           auc,
        })

    best_f1_row = max((r for r in grid_results if r['weight_scale'] == scale),
                      key=lambda r: r['f1'])
    print(f'  scale={scale:.2f}  w={w:.2f}  best F1={best_f1_row["f1"]:.4f}  '
          f'P={best_f1_row["precision"]:.3f}  R={best_f1_row["recall"]:.3f}  '
          f't={best_f1_row["threshold"]:.2f}')

grid_df = pd.DataFrame(grid_results)
print(f'\nGrid size: {len(grid_df):,} (weight × threshold) combinations')

In [ ]:
# Best F1 per weight setting
best_per_weight = grid_df.loc[grid_df.groupby('weight_scale')['f1'].idxmax()].copy()

# Best precision per weight setting (among rows where F1 >= nb51 baseline)
above_baseline = grid_df[grid_df['f1'] >= NB51_F1]
if len(above_baseline):
    best_p_above_baseline = above_baseline.loc[above_baseline.groupby('weight_scale')['precision'].idxmax()].copy()
else:
    best_p_above_baseline = pd.DataFrame()

# Overall best precision with F1 >= NB51
best_joint = above_baseline.loc[above_baseline['precision'].idxmax()] if len(above_baseline) else None
# Overall best F1
best_f1_overall = grid_df.loc[grid_df['f1'].idxmax()]

print('=== Joint Grid — Best F1 per Weight Scale ===')
print(best_per_weight[['weight_scale', 'pos_weight', 'threshold', 'f1', 'precision', 'recall', 'auc']].to_string(index=False))
print()
if best_joint is not None:
    print(f'Best precision with F1 >= nb51 baseline ({NB51_F1:.4f}):')
    print(f'  scale={best_joint.weight_scale:.2f}  t={best_joint.threshold:.2f}  '
          f'F1={best_joint.f1:.4f}  P={best_joint.precision:.3f}  R={best_joint.recall:.3f}')
print()
print(f'Best overall F1:')
print(f'  scale={best_f1_overall.weight_scale:.2f}  t={best_f1_overall.threshold:.2f}  '
      f'F1={best_f1_overall.f1:.4f}  P={best_f1_overall.precision:.3f}  R={best_f1_overall.recall:.3f}')
print()
print(f'nb51 reference:        F1={NB51_F1:.4f}  P={NB51_PRECISION:.3f}  R={NB51_RECALL:.3f}')
print(f'nb59A balanced point:  F1={NB59A_F1:.4f}  P={NB59A_P:.3f}  R={NB59A_R:.3f}')
print(f'nb59B best F1:         F1={NB59B_F1:.4f}  P={NB59B_P:.3f}  R={NB59B_R:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: P/R trade-off for each weight scale at its best-F1 threshold
ax = axes[0]
cmap = plt.cm.viridis
colors = [cmap(i / (len(weight_scales) - 1)) for i in range(len(weight_scales))]

for row, color in zip(best_per_weight.itertuples(), colors):
    ax.scatter(row.recall, row.precision, s=100, color=color, zorder=5)
    ax.annotate(f'{row.weight_scale:.2f}×',
                (row.recall, row.precision),
                textcoords='offset points', xytext=(6, 3), fontsize=7.5)

ax.scatter([NB51_RECALL], [NB51_PRECISION], color='red', s=120, marker='*',
           zorder=6, label='nb51 REF-CLEAN')
ax.scatter([NB59A_R], [NB59A_P], color='orange', s=100, marker='D',
           zorder=6, label='nb59A P=R point')

# P=R diagonal
ax.plot([0, 1], [0, 1], 'k--', alpha=0.2, lw=1)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Config A: P/R at Best-F1 Threshold per Weight Scale', fontweight='bold')
ax.legend(fontsize=8)
ax.set_xlim(0.2, 0.9)
ax.set_ylim(0.3, 0.7)
ax.grid(alpha=0.3)

# Add colorbar for weight scale
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=min(weight_scales), vmax=max(weight_scales)))
sm.set_array([])
plt.colorbar(sm, ax=ax, label='Weight scale (fraction of balanced)')

# Right: F1 heatmap over weight × threshold space (for key weight values)
ax2 = axes[1]
heat_scales = [1.0, 0.75, 0.50, 0.35, 0.25]
heat_thresholds = np.arange(0.25, 0.75, 0.05)

heat_data = []
for scale in heat_scales:
    row_data = []
    for t in heat_thresholds:
        subset = grid_df[(grid_df['weight_scale'] == scale) &
                         (grid_df['threshold'] == round(t, 2))]
        if len(subset):
            row_data.append(subset.iloc[0]['f1'])
        else:
            row_data.append(np.nan)
    heat_data.append(row_data)

heat_arr = np.array(heat_data)
im = ax2.imshow(heat_arr, aspect='auto', cmap='RdYlGn',
                vmin=max(0, np.nanmin(heat_arr) - 0.02),
                vmax=min(1, np.nanmax(heat_arr) + 0.02))

for i in range(len(heat_scales)):
    for j in range(len(heat_thresholds)):
        val = heat_arr[i, j]
        if not np.isnan(val):
            ax2.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=7)

ax2.set_xticks(range(len(heat_thresholds)))
ax2.set_xticklabels([f'{t:.2f}' for t in heat_thresholds], fontsize=8)
ax2.set_yticks(range(len(heat_scales)))
ax2.set_yticklabels([f'{s:.2f}×' for s in heat_scales], fontsize=8)
ax2.set_xlabel('Decision threshold')
ax2.set_ylabel('Weight scale')
ax2.set_title('Config A: F1 Heatmap (weight × threshold)', fontweight='bold')
plt.colorbar(im, ax=ax2, label='F1')

plt.tight_layout()
plt.savefig('../../reports/figures/60a_joint_grid.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved 60a_joint_grid.png')

## 3. Config B — FP-Driving Feature Identification

Train the baseline model and tag every test paper as TP / FP / TN / FN.
Compare venue feature means between FPs and TPs to find which features
are inflating false positives, then retrain with those features removed.

In [ ]:
# Baseline model (balanced weights, best-F1 threshold from nb59)
baseline_model = LGBMClassifier(
    n_estimators=500, learning_rate=0.05, num_leaves=63,
    class_weight='balanced', random_state=RANDOM_STATE,
    n_jobs=-1, verbose=-1
)
baseline_model.fit(X_tr, y_tr)
proba_baseline = baseline_model.predict_proba(X_te)[:, 1]

# Find best-F1 threshold
thresholds = np.arange(0.05, 0.95, 0.01)
f1s = [f1_score(y_te, (proba_baseline >= t).astype(int), zero_division=0) for t in thresholds]
best_t_baseline = thresholds[int(np.argmax(f1s))]
y_pred_baseline = (proba_baseline >= best_t_baseline).astype(int)

baseline_f1 = f1_score(y_te, y_pred_baseline)
baseline_p  = precision_score(y_te, y_pred_baseline)
baseline_r  = recall_score(y_te, y_pred_baseline)
print(f'Baseline: F1={baseline_f1:.4f}  P={baseline_p:.3f}  R={baseline_r:.3f}  t={best_t_baseline:.2f}')

# Tag outcomes
df_analysis = df_aub_test.copy()
df_analysis['y_true']  = y_te.values
df_analysis['y_pred']  = y_pred_baseline
df_analysis['proba']   = proba_baseline
df_analysis['outcome'] = 'TN'
df_analysis.loc[(df_analysis['y_true']==1) & (df_analysis['y_pred']==1), 'outcome'] = 'TP'
df_analysis.loc[(df_analysis['y_true']==0) & (df_analysis['y_pred']==1), 'outcome'] = 'FP'
df_analysis.loc[(df_analysis['y_true']==1) & (df_analysis['y_pred']==0), 'outcome'] = 'FN'

print(f"\nOutcome counts: {df_analysis['outcome'].value_counts().to_dict()}")

In [ ]:
fp = df_analysis[df_analysis['outcome'] == 'FP']
tp = df_analysis[df_analysis['outcome'] == 'TP']
tn = df_analysis[df_analysis['outcome'] == 'TN']
fn = df_analysis[df_analysis['outcome'] == 'FN']

# Feature importances from baseline model
venue_feat_names = [f for f in VENUE_FEATS if f in X_tr.columns]
imp = pd.Series(baseline_model.feature_importances_, index=X_tr.columns)
venue_imp = imp[venue_feat_names].sort_values(ascending=False)

print('=== Venue feature importances (baseline model) ===')
print(venue_imp.to_string())
print()

# FP vs TP comparison: for each venue feature, compute mean difference
raw_tr_vf = extract_venue_features(df_aub_train)
raw_te_vf = extract_venue_features(df_aub_test)
tr_med = raw_tr_vf.median()
raw_te_vf = raw_te_vf.fillna(tr_med)

fp_idx = df_analysis[df_analysis['outcome'] == 'FP'].index
tp_idx = df_analysis[df_analysis['outcome'] == 'TP'].index
tn_idx = df_analysis[df_analysis['outcome'] == 'TN'].index

fp_vf = raw_te_vf.loc[fp_idx]
tp_vf = raw_te_vf.loc[tp_idx]
tn_vf = raw_te_vf.loc[tn_idx]

compare_rows = []
for feat in venue_feat_names:
    fp_mean = fp_vf[feat].mean()
    tp_mean = tp_vf[feat].mean()
    tn_mean = tn_vf[feat].mean()
    # FP inflation: how much higher are FP means than TN means on this feature?
    fp_inflation = fp_mean - tn_mean   # positive = FPs look like TPs on this feature
    fp_tp_gap    = fp_mean - tp_mean   # negative = FPs lower than TPs (good distinction)
    compare_rows.append({
        'feature':      feat,
        'importance':   venue_imp.get(feat, 0),
        'FP_mean':      fp_mean,
        'TP_mean':      tp_mean,
        'TN_mean':      tn_mean,
        'FP_inflation': fp_inflation,  # FP - TN: high = FPs are inflated on this feature
        'FP_TP_gap':    fp_tp_gap,     # FP - TP: near-zero = FP and TP look similar
    })

compare_df = pd.DataFrame(compare_rows).sort_values('FP_inflation', ascending=False)
print('=== FP Feature Analysis (sorted by FP inflation = FP_mean - TN_mean) ===')
print(compare_df[['feature', 'importance', 'FP_mean', 'TP_mean', 'TN_mean',
                   'FP_inflation', 'FP_TP_gap']].to_string(index=False, float_format='{:.3f}'.format))
print()
print('FP_inflation > 0: FPs have higher feature value than TNs (FP-driving)')
print('FP_TP_gap ≈ 0:    FPs look identical to TPs on this feature (model confused)')

In [ ]:
# Identify FP-driving features: high inflation AND high importance
# Score = importance × FP_inflation (normalised)
compare_df['norm_importance']   = compare_df['importance'] / compare_df['importance'].max()
compare_df['norm_fp_inflation'] = compare_df['FP_inflation'].clip(lower=0)
if compare_df['norm_fp_inflation'].max() > 0:
    compare_df['norm_fp_inflation'] /= compare_df['norm_fp_inflation'].max()
compare_df['fp_drive_score'] = compare_df['norm_importance'] * compare_df['norm_fp_inflation']
compare_df = compare_df.sort_values('fp_drive_score', ascending=False)

print('=== FP-driving score (importance × FP_inflation) ===')
print(compare_df[['feature', 'importance', 'FP_inflation', 'fp_drive_score']].to_string(index=False, float_format='{:.3f}'.format))

# Select top FP-driving features to drop (top 3 by score)
top_fp_features = compare_df.head(3)['feature'].tolist()
print(f'\nTop FP-driving features to suppress: {top_fp_features}')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
x = range(len(compare_df))
w = 0.25
ax.bar([xi - w for xi in x], compare_df['FP_mean'],  w, label='FP (false positives)', color='#E25F5F', alpha=0.85)
ax.bar([xi     for xi in x], compare_df['TP_mean'],  w, label='TP (true positives)',  color='#2ca02c', alpha=0.85)
ax.bar([xi + w for xi in x], compare_df['TN_mean'],  w, label='TN (true negatives)',  color='#4878cf', alpha=0.85)
ax.set_xticks(list(x))
ax.set_xticklabels(compare_df['feature'], rotation=30, ha='right', fontsize=8)
ax.set_title('Config B: Venue Feature Means by Outcome', fontweight='bold')
ax.set_ylabel('Mean feature value')
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

ax2 = axes[1]
colors_b = ['#E25F5F' if f in top_fp_features else '#4878cf' for f in compare_df['feature']]
ax2.barh(compare_df['feature'], compare_df['fp_drive_score'], color=colors_b, alpha=0.85)
ax2.set_title('FP-Driving Score (importance × FP inflation)', fontweight='bold')
ax2.set_xlabel('Score')
ax2.invert_yaxis()
ax2.grid(axis='x', alpha=0.3)

from matplotlib.patches import Patch
ax2.legend(handles=[Patch(color='#E25F5F', label='Selected for suppression'),
                    Patch(color='#4878cf', label='Kept')], fontsize=8)

plt.tight_layout()
plt.savefig('../../reports/figures/60b_fp_feature_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved 60b_fp_feature_analysis.png')

## 4. Config B (cont.) — Retrain with FP-Driving Features Dropped

In [ ]:
suppression_configs = [
    ('drop_top1', compare_df.head(1)['feature'].tolist()),
    ('drop_top2', compare_df.head(2)['feature'].tolist()),
    ('drop_top3', compare_df.head(3)['feature'].tolist()),
    ('drop_top5', compare_df.head(5)['feature'].tolist()),
]

suppression_results = []

for label, drop_cols in suppression_configs:
    X_tr_drop, X_te_drop = build_features(df_aub_train, df_aub_test, drop_cols=drop_cols)

    model_drop = LGBMClassifier(
        n_estimators=500, learning_rate=0.05, num_leaves=63,
        class_weight='balanced', random_state=RANDOM_STATE,
        n_jobs=-1, verbose=-1
    )
    model_drop.fit(X_tr_drop, y_tr)
    proba_drop = model_drop.predict_proba(X_te_drop)[:, 1]

    # Best F1 threshold
    f1s_drop = [f1_score(y_te, (proba_drop >= t).astype(int), zero_division=0) for t in thresholds]
    best_t_drop = thresholds[int(np.argmax(f1s_drop))]
    y_pred_drop = (proba_drop >= best_t_drop).astype(int)
    auc_drop    = roc_auc_score(y_te, proba_drop)

    res = {
        'label':      label,
        'dropped':    ', '.join(drop_cols),
        'f1':         f1_score(y_te, y_pred_drop, zero_division=0),
        'precision':  precision_score(y_te, y_pred_drop, zero_division=0),
        'recall':     recall_score(y_te, y_pred_drop, zero_division=0),
        'auc':        auc_drop,
        'threshold':  best_t_drop,
    }
    suppression_results.append(res)
    print(f'  {label:12s}  F1={res["f1"]:.4f}  P={res["precision"]:.3f}  R={res["recall"]:.3f}  '
          f'AUC={res["auc"]:.4f}  dropped=[{res["dropped"]}]')

suppression_df = pd.DataFrame(suppression_results)

## 5. Config C — Combined: Best Joint Weight + FP Suppression

Take the best weight scale from Config A's grid and combine it with the
best feature-drop config from Config B.

In [ ]:
# Best weight scale: the one that maximises precision while F1 >= NB51
above_baseline = grid_df[grid_df['f1'] >= NB51_F1]
if len(above_baseline):
    best_joint_row = above_baseline.loc[above_baseline['precision'].idxmax()]
    best_joint_scale = best_joint_row['weight_scale']
    best_joint_t     = best_joint_row['threshold']
    print(f'Best joint config (P-maximising, F1 >= {NB51_F1:.4f}):')
    print(f'  weight_scale={best_joint_scale:.2f}  threshold={best_joint_t:.2f}  '
          f'F1={best_joint_row.f1:.4f}  P={best_joint_row.precision:.3f}  R={best_joint_row.recall:.3f}')
else:
    # Fall back to best-F1 overall
    best_joint_row   = grid_df.loc[grid_df['f1'].idxmax()]
    best_joint_scale = best_joint_row['weight_scale']
    best_joint_t     = best_joint_row['threshold']
    print(f'No config achieves F1 >= {NB51_F1:.4f}; using best-F1 overall:')
    print(f'  weight_scale={best_joint_scale:.2f}  threshold={best_joint_t:.2f}  '
          f'F1={best_joint_row.f1:.4f}  P={best_joint_row.precision:.3f}  R={best_joint_row.recall:.3f}')

# Best drop config (highest precision)
best_drop_row  = suppression_df.loc[suppression_df['precision'].idxmax()]
best_drop_cols = best_drop_row['dropped'].split(', ')
print(f'\nBest drop config: {best_drop_row.label}  '
      f'P={best_drop_row.precision:.3f}  F1={best_drop_row.f1:.4f}')
print(f'  dropped: {best_drop_cols}')

In [ ]:
# Combined model: best weight + best feature drop
best_w = balanced_weight * best_joint_scale
cw_combined = {0: 1.0, 1: best_w}

X_tr_comb, X_te_comb = build_features(df_aub_train, df_aub_test, drop_cols=best_drop_cols)

model_combined = LGBMClassifier(
    n_estimators=500, learning_rate=0.05, num_leaves=63,
    class_weight=cw_combined, random_state=RANDOM_STATE,
    n_jobs=-1, verbose=-1
)
model_combined.fit(X_tr_comb, y_tr)
proba_combined = model_combined.predict_proba(X_te_comb)[:, 1]

# Sweep threshold
f1s_comb   = [f1_score(y_te, (proba_combined >= t).astype(int), zero_division=0) for t in thresholds]
best_t_comb = thresholds[int(np.argmax(f1s_comb))]
y_pred_comb = (proba_combined >= best_t_comb).astype(int)

c_f1  = f1_score(y_te, y_pred_comb, zero_division=0)
c_p   = precision_score(y_te, y_pred_comb, zero_division=0)
c_r   = recall_score(y_te, y_pred_comb, zero_division=0)
c_auc = roc_auc_score(y_te, proba_combined)

print(f'Config C (combined):  F1={c_f1:.4f}  P={c_p:.3f}  R={c_r:.3f}  AUC={c_auc:.4f}  t={best_t_comb:.2f}')
print(f'  weight_scale={best_joint_scale:.2f}  dropped={best_drop_cols}')
print()
print(f'vs nb51 baseline: ΔF1={c_f1-NB51_F1:+.4f}  ΔP={c_p-NB51_PRECISION:+.3f}  ΔR={c_r-NB51_RECALL:+.3f}')

## 6. Summary Table

In [ ]:
best_grid_row = best_per_weight.loc[best_per_weight['f1'].idxmax()]
best_grid_p_row = (
    above_baseline.loc[above_baseline['precision'].idxmax()]
    if len(above_baseline) else best_grid_row
)
best_supp_row = suppression_df.loc[suppression_df['f1'].idxmax()]
best_supp_p_row = suppression_df.loc[suppression_df['precision'].idxmax()]

summary_rows = [
    {'Config': 'BASELINE (nb51)',         'F1': NB51_F1,       'Precision': NB51_PRECISION,
     'Recall': NB51_RECALL, 'AUC': NB51_AUC,
     'Notes': '75th pct, balanced weights'},
    {'Config': 'nb59A balanced P=R',       'F1': NB59A_F1,      'Precision': NB59A_P,
     'Recall': NB59A_R,     'AUC': NB51_AUC,
     'Notes': f't={NB59A_T:.2f}'},
    {'Config': 'nb59B best F1',            'F1': NB59B_F1,      'Precision': NB59B_P,
     'Recall': NB59B_R,     'AUC': None,
     'Notes': '0.5× balanced'},
    {'Config': 'A — grid best F1',         'F1': best_grid_row.f1,       'Precision': best_grid_row.precision,
     'Recall': best_grid_row.recall, 'AUC': best_grid_row.auc,
     'Notes': f'scale={best_grid_row.weight_scale:.2f}  t={best_grid_row.threshold:.2f}'},
    {'Config': 'A — grid best P (F1≥nb51)','F1': best_grid_p_row.f1,     'Precision': best_grid_p_row.precision,
     'Recall': best_grid_p_row.recall, 'AUC': best_grid_p_row.auc,
     'Notes': f'scale={best_grid_p_row.weight_scale:.2f}  t={best_grid_p_row.threshold:.2f}'},
    {'Config': f'B — {best_supp_row.label}','F1': best_supp_row.f1,      'Precision': best_supp_row.precision,
     'Recall': best_supp_row.recall, 'AUC': best_supp_row.auc,
     'Notes': f'drop [{best_supp_row.dropped}]'},
    {'Config': f'B — best P ({best_supp_p_row.label})', 'F1': best_supp_p_row.f1, 'Precision': best_supp_p_row.precision,
     'Recall': best_supp_p_row.recall, 'AUC': best_supp_p_row.auc,
     'Notes': f'drop [{best_supp_p_row.dropped}]'},
    {'Config': 'C — Combined',             'F1': c_f1,           'Precision': c_p,
     'Recall': c_r,         'AUC': c_auc,
     'Notes': f'scale={best_joint_scale:.2f}  drop [{best_drop_cols}]'},
]

summary_df = pd.DataFrame(summary_rows)
for col in ['F1', 'Precision', 'Recall', 'AUC']:
    summary_df[col] = pd.to_numeric(summary_df[col], errors='coerce').round(4)
summary_df['ΔF1'] = (summary_df['F1'] - NB51_F1).round(4)
summary_df['ΔP']  = (summary_df['Precision'] - NB51_PRECISION).round(4)

print('=' * 110)
print('NB60 RESULTS SUMMARY')
print('=' * 110)
print(summary_df[['Config', 'F1', 'Precision', 'Recall', 'AUC', 'ΔF1', 'ΔP', 'Notes']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: P/R scatter for all configs
ax = axes[0]
colors_map = {
    'BASELINE (nb51)':         'red',
    'nb59A balanced P=R':      'orange',
    'nb59B best F1':           'gold',
    'A — grid best F1':        '#4878cf',
    'A — grid best P (F1≥nb51)':'#4878cf',
}

for _, row in summary_df.iterrows():
    if pd.isna(row['Recall']) or pd.isna(row['Precision']):
        continue
    color = colors_map.get(row['Config'], '#2ca02c')
    marker = '*' if 'BASELINE' in row['Config'] else 'o'
    ax.scatter(row['Recall'], row['Precision'], s=120, color=color, marker=marker, zorder=5)
    ax.annotate(row['Config'].replace('BASELINE (nb51)', 'nb51'),
                (row['Recall'], row['Precision']),
                textcoords='offset points', xytext=(5, 3), fontsize=7)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.2, lw=1, label='P=R diagonal')
ax.axhline(NB51_PRECISION, color='red', linestyle=':', alpha=0.5, label=f'nb51 P={NB51_PRECISION}')
ax.axvline(NB51_RECALL,    color='red', linestyle=':', alpha=0.5, label=f'nb51 R={NB51_RECALL}')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('NB60: Precision-Recall Operating Points', fontweight='bold')
ax.legend(fontsize=7, loc='upper right')
ax.set_xlim(0.2, 0.9)
ax.set_ylim(0.3, 0.7)
ax.grid(alpha=0.3)

# Right: bar chart of F1, P, R for key configs
ax2 = axes[1]
key_configs = summary_df[~summary_df['Config'].str.startswith('nb59')].copy()
x = np.arange(len(key_configs))
w = 0.25
ax2.bar(x - w, key_configs['Precision'], w, label='Precision', color='#E25F5F', alpha=0.85)
ax2.bar(x,     key_configs['Recall'],    w, label='Recall',    color='#4878cf', alpha=0.85)
ax2.bar(x + w, key_configs['F1'],        w, label='F1',        color='#2ca02c', alpha=0.85)
ax2.axhline(NB51_F1,        color='#2ca02c', linestyle='--', lw=1.2, alpha=0.6, label=f'nb51 F1')
ax2.axhline(NB51_PRECISION, color='#E25F5F', linestyle='--', lw=1.2, alpha=0.6, label=f'nb51 P')
ax2.set_xticks(list(x))
ax2.set_xticklabels(key_configs['Config'], rotation=20, ha='right', fontsize=7.5)
ax2.set_ylabel('Score')
ax2.set_ylim(0, 1)
ax2.set_title('NB60: P / R / F1 by Config', fontweight='bold')
ax2.legend(fontsize=8)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../../reports/figures/60_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved 60_summary.png')

## 7. Findings & Conclusions

In [ ]:
best_config = summary_df.loc[summary_df['F1'].idxmax()]
best_p_config = summary_df.loc[summary_df['Precision'].idxmax()]

print('=' * 80)
print('NB60 FINDINGS — Joint Precision-Boost')
print('=' * 80)
print(f"""
GOAL: improve precision above nb59A balanced point ({NB59A_P:.3f}) while F1 ≥ nb51 ({NB51_F1:.4f})

CONFIG A — Joint weight × threshold grid
  Best F1:           {best_grid_row.f1:.4f}  P={best_grid_row.precision:.3f}  R={best_grid_row.recall:.3f}
  Best P (F1≥nb51):  {best_grid_p_row.f1:.4f}  P={best_grid_p_row.precision:.3f}  R={best_grid_p_row.recall:.3f}
  → Joint tuning {'improves' if best_grid_p_row.precision > NB51_PRECISION else 'does not improve'} precision vs nb51 baseline.

CONFIG B — FP feature suppression
  Top FP-driving features: {compare_df.head(3)['feature'].tolist()}
  Best precision after drop: {best_supp_p_row.precision:.3f}  F1={best_supp_p_row.f1:.4f}  ({best_supp_p_row.label})
  → Feature suppression {'helps' if best_supp_p_row.precision > NB51_PRECISION else 'does not help'} precision.

CONFIG C — Combined (best weight + best feature drop)
  F1={c_f1:.4f}  P={c_p:.3f}  R={c_r:.3f}  AUC={c_auc:.4f}
  ΔF1={c_f1-NB51_F1:+.4f}  ΔP={c_p-NB51_PRECISION:+.3f} vs nb51 baseline

OVERALL BEST
  Best F1:        {best_config.Config:40s}  F1={best_config.F1:.4f}  P={best_config.Precision:.3f}  R={best_config.Recall:.3f}
  Best Precision: {best_p_config.Config:40s}  F1={best_p_config.F1:.4f}  P={best_p_config.Precision:.3f}  R={best_p_config.Recall:.3f}
""".strip())

print()
print('RECOMMENDED NEXT STEPS')
print('─' * 60)
print(' 1. If precision ceiling is reached: accept current P/R trade-off')
print('    and focus on AUC improvements via better embeddings (SPECTER2)')
print(' 2. Try calibrated classifiers (Platt scaling / isotonic regression)')
print('    → better-calibrated probabilities may sharpen the decision boundary')
print(' 3. Stack: use baseline predictions as a meta-feature with a precision-')
print('    focused second-stage classifier trained on high-confidence FPs')